# P-Task7: API Middleware Simulator

**Problem:** backend team ko reusable function wrappers chahiye jo execution timing, input/output logging aur
role based access control handle karein, **original functions ko chhoo'e bina**. Wrappers stackable hon aur
saari activity `logging` module se ho, `print` se nahi.

**Concepts used:** closures, decorators, decorators with arguments, `functools.wraps`, stacking decorators,
`logging` (levels, named logger, file + console handler ek saath).

## 1. Logging setup
`print` ki jagah `logging` kyun? Is mein levels hote hain (DEBUG se CRITICAL tak), time khud lagta hai, aur
output ek saath screen aur file dono mein ja sakta hai. Print mein ye sab nahi milta.

| Level | Kab use karte hain |
|---|---|
| DEBUG | developer ki detail |
| INFO | normal kaam (function chala, result aaya) |
| WARNING | kuch ajeeb magar chal raha hai (function slow hai) |
| ERROR | kaam fail hua (access denied) |
| CRITICAL | poora system khatre mein |

In [1]:
import functools
import logging
import sys
import time

logger = logging.getLogger("api.middleware")        # named logger
logger.setLevel(logging.DEBUG)
logger.handlers.clear()                              # notebook dobara chalane par handlers duplicate na hon

fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S")

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)                       # screen par sirf INFO aur upar
console.setFormatter(fmt)

file_handler = logging.FileHandler("middleware.log", mode="w")
file_handler.setLevel(logging.DEBUG)                 # file mein sab kuch
file_handler.setFormatter(fmt)

logger.addHandler(console)
logger.addHandler(file_handler)
logger.propagate = False

logger.info("logging ready: console + file handler dono chal rahe hain")

11:23:05 | INFO     | logging ready: console + file handler dono chal rahe hain


## 2. Teen middleware (decorators)
Har decorator `functools.wraps` use karta hai taake original function ka naam aur docstring na badle.

In [2]:
class AccessDenied(Exception):
    pass


def timer(slow_after=0.05):
    """Decorator jo argument leta hai: function itne seconds se zyada le to WARNING."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            try:
                return func(*args, **kwargs)
            finally:
                took = time.perf_counter() - start
                if took > slow_after:
                    logger.warning("%s slow hai: %.3fs", func.__name__, took)
                else:
                    logger.debug("%s took %.4fs", func.__name__, took)
        return wrapper
    return decorator


def log_io(func):
    """Input aur output log karta hai."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info("CALL  %s args=%s kwargs=%s", func.__name__, args[1:], kwargs)
        result = func(*args, **kwargs)
        logger.info("RETURN %s -> %s", func.__name__, result)
        return result
    return wrapper


def require_role(*allowed_roles):
    """Role check. Pehla argument hamesha user dict hai: {"name":..., "role":...}"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user, *args, **kwargs):
            if user["role"] not in allowed_roles:
                logger.error("DENIED %s: %s (%s) ko %s role chahiye",
                             func.__name__, user["name"], user["role"], "/".join(allowed_roles))
                raise AccessDenied(f"{user['name']} is not allowed to call {func.__name__}")
            logger.debug("ALLOWED %s for %s", func.__name__, user["name"])
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

## 3. Original functions (middleware ke baghair, bilkul simple)
Neeche decorators sirf upar lage hain, andar ka code nahi badla.
Stacking ka order: **upar wala decorator sab se bahar** hota hai, pehle wohi chalta hai.

In [3]:
@timer(slow_after=0.05)
@log_io
@require_role("admin", "manager")
def get_all_users(user):
    """Saare users ki list."""
    return ["ayesha", "bilal", "fatima"]


@timer()
@log_io
@require_role("admin")
def delete_user(user, username):
    """Sirf admin user delete kar sakta hai."""
    return f"{username} deleted"


@timer(slow_after=0.01)
@log_io
@require_role("admin", "manager", "viewer")
def generate_report(user, month):
    """Slow report (sleep se simulate kiya)."""
    time.sleep(0.03)
    return f"report for {month}"


get_all_users.__name__, get_all_users.__doc__

('get_all_users', 'Saare users ki list.')

`functools.wraps` ki wajah se naam `get_all_users` hi raha, `wrapper` nahi ban gaya.

## 4. Test: alag alag users, alag alag functions

In [4]:
admin = {"name": "Sana", "role": "admin"}
manager = {"name": "Ali", "role": "manager"}
viewer = {"name": "Hina", "role": "viewer"}

get_all_users(admin)
generate_report(viewer, "October")
delete_user(admin, "bilal")

'bilal deleted'

In [5]:
# ye admin nahi hai, isliye AccessDenied aana chahiye
for person, action in [(manager, lambda u: delete_user(u, "fatima")), (viewer, get_all_users)]:
    try:
        action(person)
    except AccessDenied as e:
        logger.info("request blocked: %s", e)

## 5. Log file check
`print` ke bina sab kuch `middleware.log` mein bhi chala gaya (DEBUG lines ke saath).

In [6]:
with open("middleware.log") as f:
    log_lines = f.read().splitlines()

len(log_lines), log_lines[-1]

(21, '11:23:05 | INFO     | request blocked: Hina is not allowed to call get_all_users')

## Conclusion
Teeno decorators ek dusre ke upar stack ho gaye aur original functions ka code nahi badla. Saari activity
`logging` se hui: screen par INFO se upar aur file mein DEBUG tak. Access denied aur slow function ke liye
ERROR aur WARNING level alag use hui.